# Read and clean both datasets

Import and standardize columns in both datasets

In [1]:
import pandas as pd


# ──────────────────────────────────────────────────────────────────────────────
# AGREEMENT INFO – rename map
# ──────────────────────────────────────────────────────────────────────────────
AGREEMENT_INFO_RENAMES: dict[str, str] = {
    "c_year": "c_year",

    "type_accociated_document": "type_associated_document",
    "c_agreement_nature": "c_agreement_nature",

    "nuclear_weapons_free_zones": "nuclear_weapons_free_zone",
    "c_region_specific": "c_region_specific",

    "pdf_full_text": "pdf_full_text",
    "c_pdf_full_text": "c_pdf_full_text",

    "format": "agreement_format",
    "c_format": "c_agreement_format",

    "adoption": "adoption_resolution",
    "c_adoption": "c_adoption_resolution",
    "c_forum_adoption": "c_adoption_forum",

    "agreement_reached": "reached_agreement_date",
    "c_agreement_reached": "c_reached_agreement_date",
    "agreement_published_date": "agreement_published_date",

    "signed_signature_subject_to_ratification": "open_for_signature_subject_to_ratification",
    "c_signed_signature_subject_to_ratification": "c_open_for_signature_subject_to_ratification",
    "agreement_location": "signature_location",
    "c_signature_location": "c_signature_location",

    "c_agreement_pariticipants": "c_agreement_participants",
    "agreement_pariticipants_alpha": "agreement_participants_alpha",

    "c_obligations_weapon_specfic": "c_obligations_weapons_specific",

    "Instrument_reservations_deposited": "agreement_reservations_deposited",
    "c_agreement_subject_to_reservation": "c_agreement_subject_to_reservations",

    "amendment": "amendments",
    "c_amendment": "c_amendments",

    # ⚠ handled safely (swap)
    "obligations_location": "obligations_location_specific",
    "obligations_location_specified": "obligations_location",

    # ⚠ handled safely (swap)
    "obligations_domain": "obligations_domain_specific",
    "obligations_domain_specific": "obligations_domain",

    "c_obligations_location": "c_obligations_location",
    "c_obligations_domain": "c_obligations_domain",

    "obligations_weapon_specific": "obligations_weapons_specific",
    "weapon_specific_obligations_nr": "weapons_items_nr",
    "weapons_items": "weapons_items_specified",
    "weapons_items_topic": "weapons_item_topic",
    "c_weapons_items": "c_weapons_items_specified",

    "obligations_facility": "obligations_facilities",
    "c_obligations_facility": "c_obligations_facilities",
    "facility_nr": "obligations_facilities_nr",
    "c_facility_nr": "c_obligations_facilities_nr",
    "facility_specified": "obligations_facilities_specified",
    "c_obligations_facility_specified": "c_obligations_facilities_specified",

    "general_infromation_keeping": "general_information_keeping",
    "general_infromation_keeping_specified": "general_information_keeping_specified",
    "c_general_infromation_keeping": "c_general_information_keeping",

    "general_national_authorities_adoption": "general_domestic_authorities_adoption",
    "general_national_authorities_adoption_specified": "general_domestic_authorities_adoption_specified",
    "c_general_national_authorities_adoption": "c_general_domestic_authorities_adoption",
    "general_national_authorities_adoption_timeline": "general_domestic_authorities_adoption_timeline",
    "general_national_authorities_adoption_timeline_specified":
        "general_domestic_authorities_adoption_timeline_specified",
    "c_general_national_authorities_adoption_timeline":
        "c_general_domestic_authorities_adoption_timeline",

    "general_training": "general_training_experience_exchange",
    "general_training_specified": "general_training_experience_exchange_specified",
    "c_general_training": "c_general_training_experience_exchange",
    "general_training_timeline": "general_training_experience_exchange_timeline",
    "general_training_timeline_specified": "general_training_experience_exchange_timeline_specified",
    "c_general_training_timeline": "c_general_training_experience_exchange_timeline",

    "general_obligation_other": "general_obligations_other",
    "general_obligation_other_specified": "general_obligations_other_specified",
    "c_general_obligation_other": "c_general_obligations_other",
    "general_obligation_other_timeline": "general_obligations_other_timeline",
    "general_obligation_other_timeline_specified": "general_obligations_other_timeline_specified",
    "c_general_obligation_other_timeline": "c_general_obligations_other_timeline",

    "c_generla_communication": "c_general_communication",

    "agreement_association_utlilized": "agreement_association_utilized",
    "c_agreement_association_utlilized": "c_agreement_association_utilized",
    "c_agreement_association_established ": "c_agreement_association_established",
}


# ──────────────────────────────────────────────────────────────────────────────
# WEAPONS & FACILITIES – rename map
# ──────────────────────────────────────────────────────────────────────────────
WEAPONS_FACILITIES_RENAMES: dict[str, str] = {
    "Item_letter": "item_letter",
    "Item_number": "item_number",

    "item": "weapon_item",
    "subcategory": "weapon_subcategory",
    "subcategory_main": "weapon_subcategory_main",

    "ban_possession": "ban_possession_distinguishes_deployed_nondeployed",
    "c_ban_possession": "c_ban_possession_distinguishes_deployed_nondeployed",

    "ban_station": "ban_stationing",
    "c_ban_station": "c_ban_stationing",

    "ban_disposal_interaction_type": "ban_disposal_execution_type",

    "testing_restriction": "restriction_testing",
    "c_testing_restriction": "c_restriction_testing",

    "restriction_possession": "restriction_possession_distinguishes_deployed_nondeployed",
    "c_restriction_possession": "c_restriction_possession_distinguishes_deployed_nondeployed",

    "eliminitation": "elimination",
    "c_elimination": "c_elimination",

    "excepetions": "exceptions",
    "excepetions_specified": "exceptions_specified",
    "c_excpetions": "c_exceptions",

    "point_of_elimination": "elimination_point",
    "c_point_of_elimination": "c_elimination_point",
    "point_of_conversion": "conversion_point",
    "c_point_of_conversion": "c_conversion_point",
}


# ──────────────────────────────────────────────────────────────────────────────
# CORE UTILITIES
# ──────────────────────────────────────────────────────────────────────────────
def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    return df


def safe_rename(df: pd.DataFrame, renames: dict[str, str]) -> pd.DataFrame:
    """
    Rename columns safely, preventing overwrite in swap cases.
    """
    df = clean_column_names(df)

    # Step 1: temporary rename
    temp_map = {k: f"__tmp__{v}" for k, v in renames.items()}
    df = df.rename(columns=temp_map)

    # Step 2: final rename
    final_map = {f"__tmp__{v}": v for v in renames.values()}
    df = df.rename(columns=final_map)

    # Step 3: warn about duplicates
    dupes = df.columns[df.columns.duplicated()].tolist()
    if dupes:
        print(f"⚠ Duplicate columns after rename: {dupes}")
    else:
        print("✓ No duplicate columns")

    return df


def drop_duplicate_marker_cols(df: pd.DataFrame, marker: str = "_duplicate_drop") -> pd.DataFrame:
    cols_to_drop = [c for c in df.columns if c.endswith(marker)]
    if cols_to_drop:
        print(f"Dropping duplicate columns: {cols_to_drop}")
        df = df.drop(columns=cols_to_drop)
    return df


# ──────────────────────────────────────────────────────────────────────────────
# PUBLIC CLEANING FUNCTIONS
# ──────────────────────────────────────────────────────────────────────────────
def clean_agreements(df: pd.DataFrame) -> pd.DataFrame:
    df_before = df.copy()

    df = safe_rename(df, AGREEMENT_INFO_RENAMES)

    generate_rename_report(
        df_before,
        df,
        AGREEMENT_INFO_RENAMES,
        label="Agreement Info"
    )

    return df


def clean_weapons_facilities(df: pd.DataFrame) -> pd.DataFrame:
    df_before = df.copy()

    df = safe_rename(df, WEAPONS_FACILITIES_RENAMES)
    df = drop_duplicate_marker_cols(df)

    generate_rename_report(
        df_before,
        df,
        WEAPONS_FACILITIES_RENAMES,
        label="Weapons & Facilities"
    )

    return df

# Automatic report
def generate_rename_report(
    df_before: pd.DataFrame,
    df_after: pd.DataFrame,
    rename_map: dict[str, str],
    label: str = "Dataset"
) -> None:
    """
    Print a detailed report of column renaming effects.
    """

    before_cols = list(df_before.columns)
    after_cols = list(df_after.columns)

    before_set = set(before_cols)
    after_set = set(after_cols)
    rename_keys = set(rename_map.keys())
    rename_values = set(rename_map.values())

    # --- renamed columns (that actually existed)
    renamed = {
        old: rename_map[old]
        for old in before_cols
        if old in rename_map and rename_map[old] != old
    }

    # --- unchanged columns
    unchanged = [c for c in before_cols if c not in rename_map]

    # --- expected but missing
    missing_expected = [k for k in rename_keys if k not in before_set]

    # --- new columns after rename
    new_columns = list(after_set - before_set)

    # --- dropped columns
    dropped_columns = list(before_set - after_set)

    # --- duplicates after rename
    duplicates = [c for c in after_cols if after_cols.count(c) > 1]
    duplicates = list(set(duplicates))

    print("\n" + "=" * 70)
    print(f"RENAME REPORT → {label}")
    print("=" * 70)

    print(f"\n• Total columns before: {len(before_cols)}")
    print(f"• Total columns after : {len(after_cols)}")

    print(f"\n✔ Renamed columns ({len(renamed)}):")
    for old, new in renamed.items():
        print(f"  {old} → {new}")

    print(f"\n➖ Unchanged columns ({len(unchanged)}):")
    print(f"  (showing first 10) {unchanged[:10]}")

    print(f"\n⚠ Missing expected columns ({len(missing_expected)}):")
    print(f"  {missing_expected}")

    print(f"\n🆕 New columns after rename ({len(new_columns)}):")
    print(f"  {new_columns}")

    print(f"\n❌ Dropped columns ({len(dropped_columns)}):")
    print(f"  {dropped_columns}")

    if duplicates:
        print(f"\n🚨 Duplicate columns after rename:")
        print(f"  {duplicates}")
    else:
        print(f"\n✓ No duplicate columns after rename")

    print("\n" + "=" * 70 + "\n")
######


# ──────────────────────────────────────────────────────────────────────────────
# USAGE
# ──────────────────────────────────────────────────────────────────────────────
agreements_df = pd.read_csv("data/amcdata_agreement_info_V2.csv", encoding="latin-1")
weapons_df = pd.read_csv("data/amcdata_weapons_facilities_V2.csv", encoding="latin-1")

agreements_df = clean_agreements(agreements_df)
weapons_df = clean_weapons_facilities(weapons_df)


✓ No duplicate columns

RENAME REPORT → Agreement Info

• Total columns before: 271
• Total columns after : 271

✔ Renamed columns (59):
  type_accociated_document → type_associated_document
  nuclear_weapons_free_zones → nuclear_weapons_free_zone
  format → agreement_format
  c_format → c_agreement_format
  adoption → adoption_resolution
  c_adoption → c_adoption_resolution
  c_forum_adoption → c_adoption_forum
  agreement_reached → reached_agreement_date
  c_agreement_reached → c_reached_agreement_date
  signed_signature_subject_to_ratification → open_for_signature_subject_to_ratification
  c_signed_signature_subject_to_ratification → c_open_for_signature_subject_to_ratification
  agreement_location → signature_location
  c_agreement_pariticipants → c_agreement_participants
  agreement_pariticipants_alpha → agreement_participants_alpha
  c_agreement_subject_to_reservation → c_agreement_subject_to_reservations
  Instrument_reservations_deposited → agreement_reservations_deposited
  am

Clean weapons dataset

In [2]:
# # clean 'weapon_items' (string)
# weapons_df['weapon_item'] = weapons_df['weapon_item'].str.strip().str.lower()
# weapons_df = weapons_df[weapons_df['weapon_item'].notna() & (weapons_df['weapon_item'] != '')]
# weapons_df['weapon_item'] = weapons_df['weapon_item'].str.replace(',', " ")
# weapons_df['weapon_item'] = weapons_df['weapon_item'].str.replace(r's$', '', regex=True)
# weapons_df = weapons_df[weapons_df['summary_category'] != 1]


# automatic report
def report_item_cleaning(df_before: pd.DataFrame, df_after: pd.DataFrame) -> None:
    """
    Report changes after cleaning the 'weapon_item' column.
    """

    before_items = df_before['weapon_item']
    after_items = df_after['weapon_item']

    # --- Row counts
    before_rows = len(df_before)
    after_rows = len(df_after)

    # --- Missing / empty
    before_missing = before_items.isna().sum() + (before_items == '').sum()
    after_missing = after_items.isna().sum() + (after_items == '').sum()

    # --- Unique values
    before_unique = before_items.nunique(dropna=True)
    after_unique = after_items.nunique(dropna=True)

    # --- Value changes (only comparable rows)
    aligned = df_before.loc[df_after.index, 'weapon_item']
    changed_mask = aligned != after_items
    changed_count = changed_mask.sum()

    # --- Top changes (examples)
    changes = pd.DataFrame({
        "before": aligned[changed_mask],
        "after": after_items[changed_mask]
    }).drop_duplicates().head(10)

    print("\n" + "=" * 60)
    print("ITEM CLEANING REPORT")
    print("=" * 60)

    print(f"\nRows before: {before_rows}")
    print(f"Rows after : {after_rows}")
    print(f"Rows dropped: {before_rows - after_rows}")

    print(f"\nMissing/empty before: {before_missing}")
    print(f"Missing/empty after : {after_missing}")

    print(f"\nUnique items before: {before_unique}")
    print(f"Unique items after : {after_unique}")

    print(f"\nValues changed (same rows): {changed_count}")

    print("\nSample changes (before → after):")
    if not changes.empty:
        for _, row in changes.iterrows():
            print(f"  {row['before']} → {row['after']}")
    else:
        print("  No value changes detected")

    print("\n" + "=" * 60 + "\n")
#######
# copying the dataframe before it is altered
df_before = weapons_df.copy()

# function to remove weapon_item rows containing "summary"
def filter_summary_rows(df, column_name):
    pattern = r'(?i)\b(summary|summaries)\b'
    mask = df[column_name].str.contains(pattern, na=False)
    return df[~mask]

# --- cleaning ---
weapons_df['weapon_item'] = weapons_df['weapon_item'].str.strip().str.lower()
weapons_df = weapons_df[weapons_df['weapon_item'].notna() & (weapons_df['weapon_item'] != '')]
weapons_df['weapon_item'] = weapons_df['weapon_item'].str.replace(',', " ")
weapons_df['weapon_item'] = weapons_df['weapon_item'].str.replace(r's$', '', regex=True)
# filtering out "summary" rows requires two steps:
weapons_df = weapons_df[weapons_df['summary_category'] != 1]
weapons_df = filter_summary_rows(weapons_df, "weapon_item")

# --- report ---
report_item_cleaning(df_before, weapons_df)


ITEM CLEANING REPORT

Rows before: 434
Rows after : 330
Rows dropped: 104

Missing/empty before: 18
Missing/empty after : 0

Unique items before: 240
Unique items after : 217

Values changed (same rows): 330

Sample changes (before → after):
  Weapons General → weapons general
  Nuclear Weapons → nuclear weapon
  Radioactive Waste → radioactive waste
  Fissile Material → fissile material
  Nuclear Weapons  → nuclear weapon
  Nuclear Material → nuclear material
  Objects Carrying Nuclear Weapons → objects carrying nuclear weapon
  Weapons of Mass Destruction → weapons of mass destruction
  Other Weapons of Mass Destruction → other weapons of mass destruction
  Ships → ship




/tmp/ipykernel_3632/3614349730.py:72: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df[column_name].str.contains(pattern, na=False)


Standardizing empty values to NaN

In [3]:
# automatic report
def report_missing_replacement(df_before: pd.DataFrame, df_after: pd.DataFrame) -> None:
    """
    Accurate report of placeholder → NaN replacement.
    """

    # --- ensure same structure
    if not df_before.columns.equals(df_after.columns):
        raise ValueError("Columns do not match between before/after DataFrames")

    # --- total missing
    before_missing_total = df_before.isna().sum().sum()
    after_missing_total = df_after.isna().sum().sum()
    total_replaced = after_missing_total - before_missing_total

    # --- per-column diff
    before_missing_col = df_before.isna().sum()
    after_missing_col = df_after.isna().sum()
    diff_missing_col = (after_missing_col - before_missing_col)

    changed_cols = diff_missing_col[diff_missing_col > 0].sort_values(ascending=False)

    print("\n" + "=" * 60)
    print("MISSING VALUE REPLACEMENT REPORT")
    print("=" * 60)

    print(f"\nTotal missing before: {before_missing_total}")
    print(f"Total missing after : {after_missing_total}")
    print(f"Total values replaced → NaN: {total_replaced}")

    print(f"\nColumns affected ({len(changed_cols)}):")
    if not changed_cols.empty:
        for col, val in changed_cols.items():
            print(f"  {col}: +{val}")
    else:
        print("  No changes detected")

    # --- improved placeholder detection
    placeholders = ['99', '-99', 'n/a', 'na', '', 99, -99]

    df_str = df_before.astype(str).apply(lambda col: col.str.strip().str.lower())

    placeholder_counts = {}
    for val in placeholders:
        count = (df_str == val).sum().sum()
        if count > 0:
            placeholder_counts[val] = int(count)

    print("\nPlaceholder counts before replacement:")
    if placeholder_counts:
        for k, v in placeholder_counts.items():
            print(f"  '{k}': {v}")
    else:
        print("  None found")

    print("\n" + "=" * 60 + "\n")

###

# Standardize missing value representations
weapons_df_before = weapons_df.copy()
agreements_df_before = agreements_df.copy()

import numpy as np
weapons_df.replace([99, '99', 'N/A', '', ' ', '-99', -99], np.nan, inplace=True)
agreements_df.replace([99, '99', 'N/A', '', ' ', '-99', -99], np.nan, inplace=True)

report_missing_replacement(weapons_df_before, weapons_df)
report_missing_replacement(agreements_df_before, agreements_df)



MISSING VALUE REPLACEMENT REPORT

Total missing before: 112526
Total missing after : 112663
Total values replaced → NaN: 137

Columns affected (1):
  weapon_item_definition: +137

Placeholder counts before replacement:
  '99': 137



MISSING VALUE REPLACEMENT REPORT

Total missing before: 21680
Total missing after : 21766
Total values replaced → NaN: 86

Columns affected (10):
  c_title_short: +46
  adoption_forum: +16
  adoption_resolution: +11
  signature_location: +5
  states_parties_total: +2
  c_adoption_forum: +2
  depositary: +1
  c_entered_into_force: +1
  entry_into_force_date: +1
  agreement_duration: +1

Placeholder counts before replacement:
  '99': 1
  '-99': 74
  'n/a': 16




(optional) Dropping columns not relevant to our research purposes

In [4]:
# colunas_alvo = [
#     'ban_development',
#     'ban_testing',
#     'ban_production',
#     'ban_acquisition',
#     'ban_possession',
#     'ban_station',
#     'ban_transfer',
#     'ban_use',
#     'ban_disposal',
#     'restriction_development',
#     'testing_restriction',       # note: inconsistent naming in the dataset
#     'restriction_production',
#     'restriction_acquisition',
#     'restriction_possession',
#     'restriction_transfer',
#     'restriction_use',
#     'restriction_disposal',
#     'obligations_timeframe_type'
# ]

# outras_colunas = [
#     'agreement_id',
#     'item_type',
#     'item',
#     'weapon_item_definition',
#     'c_weapon_item_definition',
#     'timeframe_set_time',
#     'timeframe_other',
#     'c_obligations_timeframe',
#     'timeframe_phases',
#     'c_timeframe_phases',
#     'c_phases',
#     'eliminitation',             # note: typo in the dataset
#     'conversion',
#     'modernization',
#     'facility_destruction'
# ]

# all_columns_to_keep = outras_colunas + colunas_alvo

# df = df[all_columns_to_keep]

Save cleaned dataframes into new .csv files



In [5]:
import os

# your desired folder
folder_path = "cleaned_data"

# create folder if it doesn't exist
os.makedirs(folder_path, exist_ok=True)

# save file
file_path = os.path.join(folder_path, "weapons_cleaned.csv")
weapons_df.to_csv(file_path, index=False)

file_path = os.path.join(folder_path, "agreements_cleaned.csv")
agreements_df.to_csv(file_path, index=False)

# Classifying weapon_items as dual-use or not, and adding this classification to weapons_cleaned.csv

From now on, please refer to the datasets as:
weapons_cleaned_df
agreements_cleaned_df

## Classifying items as dual-use or not

In [19]:
# # reading the file:
# weapons_cleaned_df = pd.read_csv("cleaned_data/weapons_cleaned.csv", encoding="latin-1")

# # identifying all unique items in the 'weapon_item' column
# unique_items_lst = list(weapons_cleaned_df['weapon_item'].unique())
# print(f"Unique items in 'weapon_item' column: {(unique_items_lst)}")

# # capturing item_lst and, if available, the 'weapon_item_definition' for each item; creating a dictionary mapping 'weapon_item' to 'weapon_item_definition'
# weapon_and_definition_dict = weapons_cleaned_df.set_index('weapon_item')['weapon_item_definition'].to_dict()
# print("\nWeapon items and their definitions:")
# for weapon_item in weapon_and_definition_dict:
#     print(f"  {weapon_item}: {weapon_and_definition_dict[weapon_item]}")

# # save the dictionary to a CSV file
# weapon_def_df = pd.DataFrame(list(weapon_and_definition_dict.items()), columns=['weapon_item', 'weapon_item_definition'])
# weapon_def_df.to_csv(os.path.join(folder_path, "weapon_items_and_definitions.csv"), index=False)

# read data
df = pd.read_csv("cleaned_data/weapons_cleaned.csv", encoding="latin-1")

# keep only relevant columns
df = df[['weapon_item', 'weapon_item_definition']]

# # drop rows without weapon_item
# df = df.dropna(subset=['weapon_item'])

# replace NaN definitions with empty string
df['weapon_item_definition'] = df['weapon_item_definition'].fillna("No description")

# # OPTIONAL: strip whitespace
# df['weapon_item_definition'] = df['weapon_item_definition'].str.strip()

# group and combine descriptions
combined_df = (
    df.groupby('weapon_item')['weapon_item_definition']
    .apply(lambda x: " | ".join([desc for desc in x if desc]))  # concatenate non-empty strings
    .reset_index()
)


# ensure folder exists
folder_path = "cleaned_data"
os.makedirs(folder_path, exist_ok=True)

# save to CSV
output_path = os.path.join(folder_path, "unique_weapon_items_combined_descriptions.csv")
combined_df.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"Total unique weapon items: {len(combined_df)}")


Saved to: cleaned_data/unique_weapon_items_combined_descriptions.csv
Total unique weapon items: 217


This list is to be fed to LLM models (see methodology).

Once this is done and the LLM returns a .csv file.

Rename each file according to the model, e.g.: claude_classification.csv
load it and check/rename the columns:



In [23]:
# Loading a dual-use label dataset provided by Claude:
llm_df = pd.read_csv("cleaned_data/llm_dual_use_classification.csv", encoding="latin-1")

# merging the datasets into one, with the 'weapon_item' as the key and the dual-use labels from each dataset as separate columns
combined_df = pd.read_csv("cleaned_data/unique_weapon_items_combined_descriptions.csv", encoding="latin-1")
combined_df = combined_df.merge(llm_df, on="weapon_item", how="left")

# save the combined dual-use mapping to a new CSV file
output_path = os.path.join("cleaned_data", "weapon_items_with_dual_use_labels.csv")
combined_df.to_csv(output_path, index=False)
print(f"Saved combined dual-use mapping to: {output_path}")

KeyError: 'weapon_item'

# I stopped here (17.04.2026)

In [ ]:
dual_use_mapping_df = pd.read_csv('cleaned_data/weapon_items_with_dual_use_labels.csv', encoding= 'latin-1')

df = pd.merge(df, dual_use_mapping_df, left_on = 'weapon_item', right_on = 'weapon_category', how = 'inner')

Merge description of items (weapon_item_definition and c_weapon_item_definition) for duplicated items

In [ ]:
def merge_unique_text(series):
    vals = series.replace('', pd.NA).dropna().unique()
    return ' | '.join(sorted(vals)) if len(vals) > 0 else ''

# Apply to both description columns
for col in ['weapon_item_definition', 'c_weapon_item_definition']:
    df[col] = df.groupby('item')[col].transform(merge_unique_text)

print(df.head(15))

    agreement_id  item_type                                  item  \
0             40        1.0                       weapons general   
1             50        1.0                        nuclear weapon   
2             50        3.0                     radioactive waste   
3             50        1.0                       weapons general   
4             90        1.0                        nuclear weapon   
5             90        3.0                      fissile material   
6             70        1.0                        nuclear weapon   
7             70        3.0                      nuclear material   
8          70001        1.0                        nuclear weapon   
9          70002        1.0                        nuclear weapon   
10            80        1.0       objects carrying nuclear weapon   
11            80        1.0           weapons of mass destruction   
12            80        1.0                       weapons general   
13           100        1.0  summa

Combine agreement_ids for duplicated items

In [ ]:
df['agreement_id'] = df['agreement_id'].astype(str)
df['agreement_id'] = df.groupby('item')['agreement_id'].transform(
    lambda x: ';'.join(sorted(x.replace('', pd.NA).dropna().astype(str).unique()))
)

Drop duplicates

In [ ]:
df_no_dup = df.drop_duplicates(subset=['item'], keep='first')

Generates a clean dataset without duplicates

In [ ]:
# df_exploded.to_csv('duplicates_result.csv', index=False)
df_no_dup.to_csv('duplicates_result.csv', index=False)

Split agreement_ids to match with agreement_info

In [ ]:
df = df.drop_duplicates(subset=['item'], keep='first')

df['agreement_id'] = df['agreement_id'].str.split(';')
df_exploded = df.explode('agreement_id')
df_exploded['agreement_id'] = df_exploded['agreement_id'].str.strip()

Pulled agreement_info dataset

In [ ]:
df_agr = pd.read_csv("/workspaces/amc-research-sprint-lh_gl/data/amcdata_agreement_info_V2.csv", encoding='latin-1')
df_weapons = df_exploded 

Selected which columns from each dataset to keep

In [ ]:
weapons_col = [
    'agreement_id',
    'item_type',
    'item',
    'weapon_item_definition',
    'c_weapon_item_definition',
    'ban_development',
    'ban_testing',
    'ban_production',
    'ban_acquisition',
    'ban_possession',
    'ban_station',
    'ban_transfer',
    'ban_use',
    'ban_disposal',
    'restriction_development',
    'testing_restriction',       # note: inconsistent naming in the dataset
    'restriction_production',
    'restriction_acquisition',
    'restriction_possession',
    'restriction_transfer',
    'restriction_use',
    'restriction_disposal',
    'obligations_timeframe_type',
    'timeframe_set_time',
    'timeframe_other',
    'c_obligations_timeframe',
    'timeframe_phases',
    'c_timeframe_phases',
    'c_phases',
    'weapon_category',
    'dual_use',
    'rationale'
]

df_weapons = df_weapons[weapons_col]

agr_col = [
    'agreement_id',
    'year',
    'title_full',
    'title_short',
    'description',
    'agreement_nature',
    'superseeded_agreement',
    'region',
    'region_specific',
    'c_region_specific',
    'laterality',
    'format',
    'agreement_date',
    'status',
    'adoption',
    'adoption_date',
    'signed_definitive_signature',
    'signatory_states',
    'entered_into_force',
    'entry_into_force_date',
    'agreement_participants_nr',
    'agreement_participants',
    'nr_states_parties_total',
    'agreement_subject_to_reservations',
    'Instrument_reservations_deposited',
    'reservation_states_signature',
    'amendment',
    'c_amendment',
    'amendment_decision',
    'exit',
    'exit_other',
    'exit_notification',
    'state_withdrawal_nr',
    'state_withdrawal'
]

df_agr = df_agr[agr_col]

Turn agreement_id to string

In [ ]:
df_agr['agreement_id'] = df_agr['agreement_id'].astype(str)
df_weapons['agreement_id'] = df_weapons['agreement_id'].astype(str)

Merge cleand and agreement_info datasets

In [ ]:
df = df_weapons.merge(df_agr, on='agreement_id', how='left')

Generate a new excel file with merged data

In [ ]:
df.to_csv('mergedresults.csv', index=False)